In [23]:
import sys
sys.path.append("..")

import config
from src.loader import load_reviews, load_meta
from src import preprocessing, analysis, plots, storage

# Loading reviews dataset and metadata

In [24]:
df = storage.cached(
    "reviews_raw",
    lambda: load_reviews(config.DATASET_NAME, config.SUBSET, n=200_000)
)

In [25]:
# df = load_reviews(config.DATASET_NAME, config.SUBSET, n=config.N_SAMPLES)
print("Shape (raw):", df.shape)

Shape (raw): (200000, 10)


In [26]:
df.dtypes

rating               float64
title                    str
text                     str
images                object
asin                     str
parent_asin              str
user_id                  str
timestamp              int64
helpful_vote           int64
verified_purchase       bool
dtype: object

In [27]:
df = preprocessing.clean(df)
df = preprocessing.add_features(df)
print("Shape (clean):", df.shape)

Cleaning dataframe from redundant columns and empty values
Adding text_len, title_len and timestamp columns
Shape (clean): (199954, 11)


In [28]:
df.head()

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,title_len
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True,1433,33
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True,225,39
2,5.0,Excellent!,I love these. They even come with a carry case...,B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True,469,10
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True,1089,22
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True,240,43


In [29]:
df[["rating", "helpful_vote", "text_len", "title_len"]].describe()

,rating,helpful_vote,text_len,title_len
count,199954.000000,199954.000000,199954.000000,199954.000000
mean,4.240845,1.428474,335.611891,25.222496
std,1.264924,23.951032,539.920288,18.428807
min,1.000000,0.000000,1.000000,1.000000
25%,4.000000,0.000000,63.000000,11.000000
50%,5.000000,0.000000,165.000000,20.000000
75%,5.000000,0.000000,387.000000,33.000000
max,5.000000,6386.000000,29809.000000,181.000000


In [30]:
df.columns

Index(['rating', 'title', 'text', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase', 'text_len',
       'title_len'],
      dtype='str')

In [31]:
storage.save_stage(df, "reviews_clean")

WindowsPath('C:/Users/Filip/Desktop/UMISI/Plotly-introduction/plotly-project/data/reviews_clean.parquet')

# Product metadata

Load product metadata, clean it (numeric `price`, category levels) and save it as the `meta_clean` stage for the EDA notebook.

In [32]:
df_meta = storage.cached(
    "meta_raw",
    lambda: load_meta(config.DATASET_NAME, config.META_SUBSET, n=config.N_SAMPLES)
)
df_meta = preprocessing.clean_meta(df_meta)
print("Shape (meta):", df_meta.shape)
df_meta.head()

Cleaning metadata: numeric price and category levels
Shape (meta): (5000, 18)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,category_l1,category_l2
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,NaN,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M,None,NaN,NaN,Electronics,Television & Video
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],NaN,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE,None,NaN,NaN,Electronics,Television & Video
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS,None,NaN,NaN,Electronics,Computers & Accessories
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW,None,NaN,NaN,Electronics,Wearable Technology
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O,None,NaN,NaN,Electronics,Computers & Accessories


In [33]:
storage.save_stage(df_meta, "meta_clean")

WindowsPath('C:/Users/Filip/Desktop/UMISI/Plotly-introduction/plotly-project/data/meta_clean.parquet')